# 準備と学習

T2TT、音声の前処理、Baseline、Teacher、校正、転移3条件をこのNotebookで実行します。結果の比較は [evaluate.ipynb](evaluate.ipynb) で行います。

1. 音声ZIPを作る場合はCPUで PREPARE_AUDIO_PACKS=True にします。ZIP作成はAUDIO_PACK_WORKERS（既定2、1〜4）で並列化します。
2. 学習する場合はGPUへ切り替え、Falseにして同じ設定で最初から実行します。
3. 実行する学習スイッチを選びます。学習済みT2TTは設定済みの保存先から復元できます。
4. top-p診断を見てMASK_TOP_Pを設定し、校正後に転移3条件を学習します。

切断後は同じ実験名・revision・設定で上から再実行します。CPUのZIP準備後は学習セルをスキップします。

`DATA_CONFIG` または入力先のOverrideで、コーパスルートとmanifestを指定します。実験IDは保存先の名前です。既存モデルの再利用時は、その保存先に対応する元のIDを指定してください。

In [ ]:
#@title 共通ログ：開始・逐次出力・生存通知・完了・失敗
from pathlib import Path
import json
"""Standard-library-only process logging, also embedded in Colab before install."""


def log_event(name, state, logs, detail=""):
    from datetime import datetime, timezone
    from pathlib import Path
    root = Path(logs)
    root.mkdir(parents=True, exist_ok=True)
    line = f"{datetime.now(timezone.utc).isoformat()} [{name}] {state} {detail}\n"
    with (root / "notebook-events.log").open("a", encoding="utf-8") as stream:
        stream.write(line)
        stream.flush()
    print(line, end="", flush=True)


def run_process(command, logs, *, name="command", capture_json=False, env=None, cwd=None,
                heartbeat_seconds=10., extra_record=None):
    """Stream every child line; silence triggers liveness, never fake progress.

    stdout JSON is kept apart from stderr. Nested loggers share an owned POSIX
    process group so interruption cannot strand a second training process.
    """
    from datetime import datetime, timezone
    import hashlib
    import json
    import os
    from pathlib import Path
    import queue
    import signal
    import subprocess
    import tempfile
    import threading
    import time

    if heartbeat_seconds <= 0:
        raise ValueError("heartbeat_seconds must be positive")
    root = Path(logs)
    root.mkdir(parents=True, exist_ok=True)
    command = list(map(str, command))
    key = hashlib.sha256(json.dumps(name, ensure_ascii=False).encode()).hexdigest()[:12] + "-" + str(time.time_ns())
    output, result_path, receipt = root / (key + ".log"), root / (key + ".stdout.log"), root / (key + ".json")
    launch_env = dict(os.environ if env is None else env)
    launch_env["PYTHONUNBUFFERED"] = "1"
    owns_group = os.name == "posix" and launch_env.get("AT_LOG_MANAGED_GROUP") != "1"
    if owns_group:
        launch_env["AT_LOG_MANAGED_GROUP"] = "1"
    started, clock = datetime.now(timezone.utc).isoformat(), time.perf_counter()
    record = dict(name=name, command=command, started_utc=started,
        host_utc_offset_seconds=int(datetime.now().astimezone().utcoffset().total_seconds()),
        elapsed_seconds=0., returncode=None, status="running", exception=None,
        pid=None, log=str(output.resolve()), stdout_log=str(result_path.resolve()) if capture_json else None,
        gpu_csv=None, gpu_monitor_error=None, cwd=str(Path(cwd or Path.cwd()).resolve()))
    lock, stop, done = threading.Lock(), threading.Event(), threading.Event()
    messages, reader_errors = queue.Queue(maxsize=256), []
    process = reader = result_stream = None
    stream = output.open("w", encoding="utf-8")

    def save_record():
        record["elapsed_seconds"] = time.perf_counter() - clock
        fd, temporary = tempfile.mkstemp(dir=root, prefix=".log-record-")
        try:
            with os.fdopen(fd, "w", encoding="utf-8") as target:
                json.dump(record, target, ensure_ascii=False, indent=2, allow_nan=False)
                target.write("\n")
                target.flush()
            os.replace(temporary, receipt)
        finally:
            if os.path.exists(temporary):
                os.unlink(temporary)

    def notice(state, detail=""):
        line = f"[{name}] {state} pid={record['pid']} elapsed={time.perf_counter()-clock:.1f}s {detail}\n"
        with lock:
            stream.write(line)
            stream.flush()
        print(line, end="", flush=True)

    def read_lines(pipe):
        try:
            for line in pipe:
                with lock:
                    stream.write(line)
                    stream.flush()
                while not stop.is_set():
                    try:
                        messages.put(line, timeout=.1)
                        break
                    except queue.Full:
                        pass
        except Exception as error:
            reader_errors.append(error)
        finally:
            done.set()

    def terminate_child():
        if process is None:
            return
        if owns_group:
            try:
                os.killpg(process.pid, signal.SIGTERM)
            except ProcessLookupError:
                pass
        elif process.poll() is None:
            process.terminate()
        try:
            process.wait(timeout=5)
        except subprocess.TimeoutExpired:
            if owns_group:
                os.killpg(process.pid, signal.SIGKILL)
            else:
                process.kill()
            process.wait()
        if owns_group:
            # A nested child may outlive its exited leader after cancellation.
            try:
                os.killpg(process.pid, signal.SIGKILL)
            except ProcessLookupError:
                pass

    try:
        notice("started", f"full log: {output.resolve()}")
        save_record()
        if capture_json:
            result_stream = result_path.open("w", encoding="utf-8")
        process = subprocess.Popen(command, stdout=result_stream if capture_json else subprocess.PIPE,
            stderr=subprocess.PIPE if capture_json else subprocess.STDOUT, text=True,
            encoding="utf-8", errors="replace", bufsize=1, env=launch_env, cwd=cwd,
            start_new_session=owns_group)
        record["pid"] = process.pid
        notice("running", "waiting for child output")
        save_record()
        pipe = process.stderr if capture_json else process.stdout
        reader = threading.Thread(target=read_lines, args=(pipe,), daemon=True, name="command-log-reader")
        reader.start()
        last_output = last_notice = time.perf_counter()
        while not (done.is_set() and messages.empty() and process.poll() is not None):
            try:
                line = messages.get(timeout=min(.1, heartbeat_seconds))
            except queue.Empty:
                line = None
            if line is not None:
                print(line, end="", flush=True)
                if line.strip():
                    last_output = time.perf_counter()
            if reader_errors:
                raise reader_errors[0]
            now = time.perf_counter()
            if now - max(last_output, last_notice) >= heartbeat_seconds:
                notice("running", f"no_output={now-last_output:.1f}s (liveness only; progress not confirmed)")
                save_record()
                last_notice = now
        code = process.wait()
        if reader_errors:
            raise reader_errors[0]
        if capture_json and code == 0:
            json.loads(result_path.read_text(encoding="utf-8"))
        record.update(returncode=code, status="succeeded" if code == 0 else "failed_or_interrupted")
        if code:
            raise subprocess.CalledProcessError(code, command)
        notice("completed", f"exit={code}; full log: {output.resolve()}")
    except BaseException as error:
        record.update(status="failed_or_interrupted", exception=type(error).__name__ + ": " + str(error))
        terminate_child()
        if process is not None:
            record["returncode"] = process.returncode
        notice("interrupted" if isinstance(error, KeyboardInterrupt) else "failed",
               f"exit={record['returncode']} {type(error).__name__}; full log: {output.resolve()}")
        raise
    finally:
        stop.set()
        if reader is not None:
            reader.join(timeout=2)
        if process is not None and (reader is None or not reader.is_alive()):
            pipe = process.stderr if capture_json else process.stdout
            if pipe is not None:
                pipe.close()
        if result_stream is not None:
            result_stream.close()
        if extra_record is not None:
            record.update(extra_record() if callable(extra_record) else extra_record)
        save_record()
        stream.close()
    result = subprocess.CompletedProcess(command, record["returncode"],
        result_path.read_text(encoding="utf-8") if capture_json else None)
    result.record = record
    return result


def execute(*arguments, name=None, capture_json=False, env=None, cwd=None):
    command = list(map(str, arguments))
    if name is None:
        name = command[2].rsplit('.', 1)[-1] if len(command) > 2 and command[1] == '-m' else Path(command[0]).name
        if '--method' in command:
            name += ':' + command[command.index('--method') + 1]
        if '--prior-type' in command:
            name += ':' + command[command.index('--prior-type') + 1]
    return run_process(command, LOGS, name=name, capture_json=capture_json, env=env, cwd=cwd)


def execute_json(*arguments, name=None, env=None, cwd=None):
    return json.loads(execute(*arguments, name=name, capture_json=True, env=env, cwd=cwd).stdout)

LOGS = Path('/content/alignment-transfer-logs/training/bootstrap')

In [ ]:
from google.colab import drive
log_event('drive-mount', 'started', LOGS)
try:
    drive.mount('/content/drive')
except BaseException as error:
    log_event('drive-mount', 'failed', LOGS, type(error).__name__)
    raise
else:
    log_event('drive-mount', 'completed', LOGS)
import json, os, subprocess, sys, time
from pathlib import Path


In [ ]:
#@title 実験と実行する学習
EXPERIMENT_ID = 'experiment-001' #@param {type:"string"}
PREPARE_S2UT = True #@param {type:"boolean"}
PREPARE_AUDIO_PACKS = True #@param {type:"boolean"}
AUDIO_PACK_WORKERS = 2 #@param {type:"integer"}
RUN_T2TT = False #@param {type:"boolean"}
EVALUATE_T2TT = False #@param {type:"boolean"}
RUN_BASELINE = False #@param {type:"boolean"}
RUN_TEACHER_ARTIFACTS = False #@param {type:"boolean"}
RUN_MASK_TOPP_DIAGNOSTICS = False #@param {type:"boolean"}
RUN_ALIGNMENT_CALIBRATION = False #@param {type:"boolean"}
RUN_DISTILLATION = False #@param {type:"boolean"}
RUN_ANNEALED_DISTILLATION = False #@param {type:"boolean"}
RUN_MASK_DISTILLATION = False #@param {type:"boolean"}
RUN_ALIGNMENT_DIAGNOSTICS = False #@param {type:"boolean"}
MASK_TOP_P = None # 診断表を確認後に0.8 / 0.9 / 0.95を設定
CALIBRATION_BATCHES = 16 #@param {type:"integer"}
T2TT_RUN_ID = 't2tt-001' #@param {type:"string"}
T2TT_MAX_UPDATES = 200000 #@param {type:"integer"}
T2TT_TOKENIZER = 'sentencepiece' #@param ['character', 'sentencepiece']
DATA_CONFIG = 'configs/data/colab.yaml' #@param {type:"string"}
SOURCE_MANIFEST_OVERRIDE = '' #@param {type:"string"}
RELEASE_SUMMARY_OVERRIDE = '' #@param {type:"string"}
CORPUS_ROOT_OVERRIDE = '' #@param {type:"string"}
T2TT_SOURCE_MANIFEST_OVERRIDE = '' #@param {type:"string"}
NEED_TEACHER = RUN_TEACHER_ARTIFACTS or RUN_MASK_TOPP_DIAGNOSTICS or RUN_ALIGNMENT_CALIBRATION or RUN_DISTILLATION or RUN_ANNEALED_DISTILLATION or RUN_MASK_DISTILLATION or RUN_ALIGNMENT_DIAGNOSTICS
NEED_TEXT = RUN_T2TT or EVALUATE_T2TT or NEED_TEACHER
if NEED_TEACHER and T2TT_TOKENIZER != 'sentencepiece':
    raise ValueError('主比較のTeacherにはSentencePiece T2TTを選択してください')
if not PREPARE_S2UT and (RUN_BASELINE or NEED_TEACHER):
    raise ValueError('Baseline・転移学習ではPREPARE_S2UT=Trueにしてください')

if isinstance(AUDIO_PACK_WORKERS, bool) or not isinstance(AUDIO_PACK_WORKERS, int) or not 1 <= AUDIO_PACK_WORKERS <= 4:
    raise ValueError("AUDIO_PACK_WORKERSは1〜4の整数を指定してください")


In [ ]:
#@title 保存先と固定環境
import os
import sys
for value in (EXPERIMENT_ID, globals().get('T2TT_RUN_ID', 'text')):
    if not value or Path(value).name != value or value in ('.', '..'):
        raise ValueError('実験名は単一のディレクトリ名にしてください')
REPO_URL = 'https://github.com/Yaaamashiro/alignment-transfer-s2ut.git'
REPO_REVISION = '8b8b778b97f626ef04f5aeb87d31541728c003c4'
REPO = Path('/content/alignment-transfer-s2ut')
PYTHON = '/content/alignment-venv/bin/python'
ROOT = str(Path('/content/alignment-transfer-work') / EXPERIMENT_ID)
PERSISTENT = str(Path('/content/drive/MyDrive/alignment-transfer-s2ut-data') / EXPERIMENT_ID)
LOGS = Path(ROOT) / 'execution'
BENCHMARK_RUNTIME = Path('/content/alignment-benchmark-runtime')
BENCHMARK_RUNTIME_REVISION = '0c0024ea888206149a3119bf096b86ba66d63b66'
DATA = Path(ROOT) / 'data'
COMMON = DATA / 'common'
BENCHMARK_UNITS = DATA / 's2ut/units'
PREPARED = DATA / 's2ut/fairseq'
BACKUP = Path(PERSISTENT) / 'baseline'
RESTORED = Path(ROOT) / 'baseline'
BASELINE = Path(ROOT) / 'registrations/baseline.json'
MANIFEST = Path(ROOT) / 'manifest/manifest.jsonl'
BACKBONE_CONFIG = Path(ROOT) / 'backbone.json'
SCOPE_SUFFIX = '-spm-all-layers'
RESEARCH_ROOT = Path(ROOT) / ('research-v3' + SCOPE_SUFFIX)
TOPP_ROOT = RESEARCH_ROOT / 'top-p'
TOPP_REPORT = TOPP_ROOT / 'top-p.json'
CALIBRATION_ROOT = RESEARCH_ROOT / 'calibration'
CALIBRATION = CALIBRATION_ROOT / 'alignment-calibration.json'
METHOD_LABELS = [
    ('distillation', 'KL-based Attention Distillation'),
    ('annealed_distillation', 'Annealed KL-based Attention Distillation'),
    ('mask_distillation', 'Attention-level Knowledge Distillation')]
os.environ.update(HF_HOME='/content/huggingface', MPLBACKEND='Agg', PYTHONUNBUFFERED='1', PYTHONUTF8='1',
    MPLCONFIGDIR='/content/matplotlib-cache', S2ST_DRIVE_SAFE='1',
    S2ST_BACKUP_MIN_SECONDS='600', S2ST_BACKUP_MAX_GB='1024')

## 1. 環境

共通のPython・fairseq・benchmark runtimeを構築し、GPUで少数更新・保存・再開を検査します。

In [ ]:
#@title Python・fairseq環境を構築
if not (REPO / '.git').is_dir():
    execute('git', 'clone', '--no-checkout', REPO_URL, str(REPO), name='clone-alignment')
execute('git', '-C', str(REPO), 'fetch', 'origin', REPO_REVISION, name='fetch-alignment')
execute('git', '-C', str(REPO), 'checkout', '--detach', REPO_REVISION, name='checkout-alignment')
os.chdir(REPO)
execute(sys.executable, '-m', 'pip', 'install', 'uv', name='install-uv')
execute('uv', 'venv', '--python', '3.10.22', '--seed', '--allow-existing', '/content/alignment-venv', name='create-python-environment')
for packages in [
    ['pip==23.3.2', 'setuptools<81', 'wheel'],
    ['torch==2.7.1', 'torchaudio==2.7.1', 'numpy==1.26.4', 'Cython==3.2.9'],
    ['--no-build-isolation', '-r', 'requirements-fairseq.txt'],
    ['-r', 'requirements.txt'],
]:
    execute(PYTHON, '-m', 'pip', 'install', *packages)


In [ ]:
#@title benchmark runtimeを固定revisionで導入
if not (BENCHMARK_RUNTIME / '.git').is_dir():
    execute('git', 'clone', '--no-checkout', 'https://github.com/Yaaamashiro/ja-en-direct-s2st-benchmark.git', str(BENCHMARK_RUNTIME))
execute('git', '-C', str(BENCHMARK_RUNTIME), 'fetch', 'origin', BENCHMARK_RUNTIME_REVISION, name='fetch-benchmark-runtime')
execute('git', '-C', str(BENCHMARK_RUNTIME), 'checkout', '--detach', BENCHMARK_RUNTIME_REVISION, name='checkout-benchmark-runtime')
execute(PYTHON, '-m', 'pip', 'install', 'PyYAML==6.0.2', '--no-deps', '-e', str(BENCHMARK_RUNTIME))
execute(PYTHON, '-m', 'pip', 'check', name='dependency-check')


In [ ]:

config = execute_json(PYTHON, '-c',
    "import json,sys,yaml; print(json.dumps(yaml.safe_load(open(sys.argv[1], encoding='utf-8'))))",
    DATA_CONFIG, name='data-settings')
needs_speech = globals().get('PREPARE_S2UT', globals().get('EVALUATE_S2UT', False))
root_value = CORPUS_ROOT_OVERRIDE or config.get('corpus_root')
manifest_value = SOURCE_MANIFEST_OVERRIDE or config.get('accepted_manifest')
summary_value = RELEASE_SUMMARY_OVERRIDE or config.get('release_summary')
if needs_speech and (not root_value or not manifest_value):
    raise ValueError('DATA_CONFIGまたはCORPUS_ROOT_OVERRIDE / SOURCE_MANIFEST_OVERRIDEでコーパスとmanifestを指定してください')
CORPUS_ROOT = Path(root_value) if root_value else None
SOURCE_MANIFEST = Path(manifest_value) if manifest_value else None
RELEASE_SUMMARY = Path(summary_value) if summary_value else None
if CORPUS_ROOT and any(path.resolve().is_relative_to(CORPUS_ROOT.resolve()) for path in (Path(ROOT), Path(PERSISTENT))):
    raise ValueError('出力先をコーパス原本の外に指定してください')
os.environ.pop('S2ST_INPUT_MAP', None)
execute(PYTHON, '-m', 'pip', 'install', 'editdistance==0.8.1', 'librosa==0.10.2.post1',
    'accelerate==1.12.0', 'pandas==2.2.3', 'psutil==7.2.2', name='runtime-dependencies')
execute(PYTHON, '-m', 'pip', 'check', name='dependency-check')


In [ ]:
execute(PYTHON, '-m', 'alignment_transfer_s2ut.environment', '--native', '--benchmark',
    *([] if PREPARE_AUDIO_PACKS else ['--cuda']), name='corpus-environment')
if PREPARE_AUDIO_PACKS:
    log_event('corpus-native-fp16-test', 'skipped', LOGS, 'CPU audio preparation')
else:
    execute(PYTHON, '-c',
        'from direct_s2st.s2ut.paper_recipe import preflight_dependencies; print(preflight_dependencies())',
        name='corpus-paper-dependencies')
    execute(PYTHON, '-m', 'pytest', 'tests/test_benchmark_native.py::test_native_fp16_trainer_and_fp32_prior',
        'tests/test_corpus_runtime.py::test_corpus_release_to_native_recipe',
        '-v', '-s', '-m', 'gpu', '--tb=short', '--basetemp', '/content/corpus-native-fp16-test',
        cwd=REPO, name='corpus-native-fp16-test')


In [ ]:
#@title 実行・保存・復元の共通処理
def corpus_stage(name, module, *arguments, work, restore_only=False):
    command = [PYTHON, '-m', 'alignment_transfer_s2ut.colab_runtime', 'run', '--work', str(work),
        '--backup', str(Path(PERSISTENT) / 'stages' / name)]
    if restore_only:
        command.append('--restore-only')
    execute(*command, '--', PYTHON, '-m', module, *map(str, arguments), name=name)

def corpus_common(restore_only=False):
    os.environ['CORPUS_ROOT'] = str(CORPUS_ROOT)
    summary_args = ['--summary', str(RELEASE_SUMMARY)] if RELEASE_SUMMARY else []
    corpus_stage('common', 'alignment_transfer_s2ut.benchmark_data', 'corpus',
        '--source', SOURCE_MANIFEST, '--corpus-root', CORPUS_ROOT, *summary_args,
        '--output', COMMON, '--resume', work=COMMON, restore_only=restore_only)
    lock = json.loads((COMMON / 'dataset-lock.json').read_text(encoding='utf-8'))
    expected = config.get('expected_pairs')
    if expected is not None and lock['total_pairs'] != expected:
        raise ValueError(f"train/dev count {lock['total_pairs']} differs from expected_pairs={expected}")
    print({'pairs': lock['total_pairs'], 'splits': lock['splits'],
        'estimated_waveform_GiB': round(lock['wav_bytes_estimate'] / 1024**3, 1)})

def corpus_stage_audio(create=False):
    staged = execute_json(PYTHON, '-m', 'alignment_transfer_s2ut.colab_runtime', 'stage-corpus',
        '--common', str(COMMON), '--corpus-root', str(CORPUS_ROOT), '--packs', str(Path(PERSISTENT) / 'wav-packs'),
        '--local', str(Path(ROOT) / 'staged-inputs'), *(['--create-audio', '--audio-pack-workers', str(AUDIO_PACK_WORKERS)] if create else ['--stream-audio']), name='audio-staging')
    if not create:
        os.environ['S2ST_INPUT_MAP'] = staged['input_map']
    print(staged)

def corpus_prepare(restore_only=False):
    corpus_stage('unit-extraction', 'alignment_transfer_s2ut.corpus_runtime', 'units',
        '--common', COMMON, '--output', BENCHMARK_UNITS, '--cache', '/content/alignment-assets',
        work=BENCHMARK_UNITS, restore_only=restore_only)
    corpus_stage('fairseq-data', 'alignment_transfer_s2ut.corpus_runtime', 'prepare',
        '--common', COMMON, '--units', BENCHMARK_UNITS, '--output', PREPARED,
        work=PREPARED, restore_only=restore_only)

def corpus_baseline(restore_only=False):
    execute(PYTHON, '-m', 'alignment_transfer_s2ut.corpus_runtime', 'baseline',
        '--data', str(DATA), '--work', str(RESTORED), '--backup', str(BACKUP),
        *(['--restore-only'] if restore_only else ['--train']), name='baseline')
def run(name, module, *arguments, gpu=False, env=None, restore_only=False):
    arguments = list(map(str, arguments))
    payload = [PYTHON, '-m', module, *arguments]
    # Read-only environment checks and asset downloads have no stage output.
    if '--output' in arguments and module != 'alignment_transfer_s2ut.benchmark' or (
            module == 'alignment_transfer_s2ut.benchmark' and arguments[0] != 'plan'):
        if '--output' in arguments:
            output = Path(arguments[arguments.index('--output') + 1])
            work = output.parent if output.suffix in ('.json', '.jsonl', '.png') else output
            wrapper = [PYTHON, '-m', 'alignment_transfer_s2ut.colab_runtime', 'run',
                '--work', str(work), '--backup', str(Path(PERSISTENT) / 'stages' / name)]
            if restore_only:
                wrapper.append('--restore-only')
            if module == 'alignment_transfer_s2ut.train_t2tt' or module == 'alignment_transfer_s2ut.benchmark' and arguments[0] == 'train':
                wrapper.append('--training')
            payload = [*wrapper, '--', *payload]
    command = [PYTHON, '-m', 'alignment_transfer_s2ut.pilot', 'run', '--name', name, '--logs', str(LOGS)]
    if gpu:
        command.append('--monitor-gpu')
    return execute(*command, '--', *payload, name='notebook:' + name, env=env)

## 2. T2TT

学習済みSentencePiece T2TTの保存先が既定値です。新たに学習する場合は新しいT2TT_RUN_IDを指定し、RUN_T2TT=Trueにします。EVALUATE_T2TT=Trueでdev BLEU・翻訳例・曲線を確認できます。

T2TTだけを実行する場合はPREPARE_S2UT=False、PREPARE_AUDIO_PACKS=Falseにします。

In [ ]:
T2TT_SOURCE_MANIFEST = T2TT_SOURCE_MANIFEST_OVERRIDE or config.get('teacher_source_manifest', '')
TEXT_ROOT = Path('/content/alignment-transfer-work') / T2TT_RUN_ID
TEXT_PERSISTENT = Path('/content/drive/MyDrive/alignment-transfer-s2ut-data') / T2TT_RUN_ID
TEXT_LOGS = TEXT_ROOT / 'execution'
TEXT_MANIFEST = TEXT_ROOT / 'text-manifest/manifest.jsonl'
TEXT_CONFIG = REPO / 'configs/t2tt/fisher.yaml'
TEXT_TRAIN_STAGE = 't2tt-' + T2TT_TOKENIZER
TEXT_WORK = TEXT_ROOT / TEXT_TRAIN_STAGE
TEXT_BACKUP = TEXT_PERSISTENT / 'stages' / TEXT_TRAIN_STAGE
TEXT_CHECKPOINT = TEXT_WORK / 'checkpoint_best.pt'
TEXT_EVAL_ROOT = TEXT_ROOT / ('dev-' + T2TT_TOKENIZER)
TEXT_EVAL_BACKUP = TEXT_PERSISTENT / 'stages' / ('dev-' + T2TT_TOKENIZER)
TEXT_DEV_METRICS = TEXT_EVAL_ROOT / 'dev-text.json'
T2TT_SNAPSHOT_UPDATES = 10000
T2TT_KEEP_LAST = 2
if NEED_TEXT and not T2TT_SOURCE_MANIFEST:
    raise ValueError('T2TT_SOURCE_MANIFEST_OVERRIDEまたはDATA_CONFIGのteacher_source_manifestを指定してください')


In [ ]:
if not PREPARE_AUDIO_PACKS and NEED_TEXT:
    prepared = execute_json(PYTHON, '-m', 'alignment_transfer_s2ut.text_manifest', '--input', T2TT_SOURCE_MANIFEST, '--output', str(TEXT_MANIFEST), '--resume', name='prepare-external-text')
    print(json.dumps(prepared, ensure_ascii=False, indent=2))
else:
    log_event('text-input', 'skipped', LOGS, 'stage disabled or CPU audio preparation')

In [ ]:
if not PREPARE_AUDIO_PACKS and NEED_TEXT:
    if RUN_T2TT:
        training_args = ['--manifest', str(TEXT_MANIFEST), '--config', str(TEXT_CONFIG), '--output', str(TEXT_WORK),
            '--backend', 'fairseq', '--device', 'cuda', '--tokenizer', T2TT_TOKENIZER, '--max-updates', str(T2TT_MAX_UPDATES),
            '--batch-size', '8', '--update-frequency', '4', '--max-tokens', '20000', '--loader-workers', '2',
            '--validation-interval', '1000', '--save-interval-updates', '100', '--resume-if-exists']
        if T2TT_TOKENIZER == 'sentencepiece':
            training_args += ['--source-vocab-size', '8000', '--target-vocab-size', '8000', '--sentencepiece-model-type', 'unigram']
        execute(PYTHON, '-m', 'alignment_transfer_s2ut.pilot', 'run', '--name', TEXT_TRAIN_STAGE, '--logs', str(TEXT_LOGS),
            '--monitor-gpu', '--', PYTHON, '-m', 'alignment_transfer_s2ut.text_t2tt_runtime', 'train',
            '--work', str(TEXT_WORK), '--backup', str(TEXT_BACKUP), '--snapshot-updates', str(T2TT_SNAPSHOT_UPDATES), '--keep-last', str(T2TT_KEEP_LAST),
            '--', PYTHON, '-m', 'alignment_transfer_s2ut.train_t2tt', *training_args, name='text-t2tt-training')
    else:
        log_event(TEXT_TRAIN_STAGE, 'skipped', TEXT_LOGS, 'training disabled; restore completed checkpoint below')
else:
    log_event('text-training', 'skipped', LOGS, 'stage disabled or CPU audio preparation')

In [ ]:
if not PREPARE_AUDIO_PACKS and NEED_TEXT:
    restored = execute_json(PYTHON, '-m', 'alignment_transfer_s2ut.evaluate_t2tt_external', 'restore-inputs',
        '--work', str(TEXT_WORK), '--backup', str(TEXT_BACKUP), '--expected-updates', str(T2TT_MAX_UPDATES), '--dev-backup', str(TEXT_EVAL_BACKUP), '--dev-output', str(TEXT_DEV_METRICS), name='restore-text-t2tt')
    status = execute_json(PYTHON, '-m', 'alignment_transfer_s2ut.text_t2tt_runtime', 'status', '--work', str(TEXT_WORK),
        '--target-updates', str(T2TT_MAX_UPDATES), name='verify-text-t2tt-complete')
    print(status)
else:
    log_event('text-restore', 'skipped', LOGS, 'stage disabled or CPU audio preparation')

In [ ]:
if not PREPARE_AUDIO_PACKS and EVALUATE_T2TT:
    execute(PYTHON, '-m', 'alignment_transfer_s2ut.colab_runtime', 'run', '--work', str(TEXT_EVAL_ROOT), '--backup', str(TEXT_EVAL_BACKUP), '--',
        PYTHON, '-m', 'alignment_transfer_s2ut.evaluate_t2tt', '--manifest', str(TEXT_MANIFEST), '--checkpoint', str(TEXT_CHECKPOINT),
        '--split', 'dev', '--beam-size', '5', '--length-penalty', '1', '--device', 'cuda', '--output', str(TEXT_DEV_METRICS), '--resume', name='text-dev-bleu')
    metrics = json.loads(TEXT_DEV_METRICS.read_text(encoding='utf-8'))
    print(json.dumps({k:v for k,v in metrics.items() if k not in ('evaluation_contract',)}, ensure_ascii=False, indent=2))
else:
    log_event('text-evaluation', 'skipped', LOGS, 'stage disabled or CPU audio preparation')

In [ ]:
if not PREPARE_AUDIO_PACKS and EVALUATE_T2TT:
    import pandas as pd
    from IPython.display import display, Image
    predictions = [json.loads(line) for line in TEXT_DEV_METRICS.with_name(TEXT_DEV_METRICS.stem + '-predictions.jsonl').read_text(encoding='utf-8').splitlines() if line.strip()]
    references = {row['pair_id']:row['en_text'] for row in (json.loads(line) for line in TEXT_MANIFEST.read_text(encoding='utf-8').splitlines() if line.strip()) if row['split']=='dev'}
    display(pd.DataFrame([{'pair_id':r['pair_id'], 'JA':r['source_text'], 'Reference EN':references[r['pair_id']],
        'T2TT EN':r['hypothesis'], 'EOS':r['eos_reached']} for r in predictions[:10]]))
    diagnostic = TEXT_WORK / 'tokenization.json'
    if diagnostic.is_file():
        tokenization = json.loads(diagnostic.read_text(encoding='utf-8'))
        print('actual vocab sizes:', tokenization['source']['actual_vocab_size'], tokenization['target']['actual_vocab_size'])
        display(pd.DataFrame(tokenization['statistics']).T)
    execute(PYTHON, '-m', 'alignment_transfer_s2ut.text_t2tt_runtime', 'report', '--work', str(TEXT_WORK),
        '--output', str(TEXT_WORK / 'diagnostics'), name='text-learning-curve')
    display(Image(filename=str(TEXT_WORK / 'diagnostics/learning-curve.png')))
else:
    log_event('text-examples', 'skipped', LOGS, 'stage disabled or CPU audio preparation')

## 3. 音声と学習データ

QC合格のtrain/dev、WAV ZIP、HuBERT L6/KM100 unit、辞書、fairseqデータを準備します。GPUでは音声ZIPの索引を作り、使用する分から取得・検証します。初回はunitと辞書を準備します。再開時は保存済みの前処理とcheckpointを復元し、データローダーが先のバッチを取得します。

In [ ]:
if PREPARE_S2UT:
    corpus_common()
    corpus_stage_audio(create=PREPARE_AUDIO_PACKS)
    if PREPARE_AUDIO_PACKS:
        print('WAV ZIPの準備完了。GPUに切り替え、PREPARE_AUDIO_PACKS=Falseで上から再実行してください。')
else:
    log_event('audio-staging', 'skipped', LOGS, 'PREPARE_S2UT=False')


In [ ]:
if PREPARE_S2UT:
    if PREPARE_AUDIO_PACKS:
        log_event('corpus-s2ut-preparation', 'skipped', LOGS, 'CPU audio preparation')
    else:
        corpus_prepare()
        execute(PYTHON, '-c',
            'import sys; from alignment_transfer_s2ut.benchmark import prepared_identity; '
            'print(prepared_identity(sys.argv[1])["data_lock"]["splits"])',
            str(PREPARED), name='corpus-registration-preflight')
else:
    log_event('speech-preparation', 'skipped', LOGS, 'PREPARE_S2UT=False')


## 4. Baseline

RUN_BASELINE=Trueで400,000 updatesを学習・再開します。Teacherや転移学習へ進む場合は、完了したBaselineを復元して登録します。

In [ ]:
if PREPARE_AUDIO_PACKS:
    log_event('baseline', 'skipped', LOGS, 'CPU audio preparation')
elif RUN_BASELINE or NEED_TEACHER:
    corpus_baseline(restore_only=not RUN_BASELINE)
else:
    log_event('baseline', 'skipped', LOGS, 'training and teacher stages disabled')

In [ ]:
if not PREPARE_AUDIO_PACKS and (RUN_BASELINE or NEED_TEACHER):
    run('register-baseline', 'alignment_transfer_s2ut.benchmark', 'register',
        '--checkpoint', RESTORED / 'checkpoints/checkpoint_best.pt', '--data', PREPARED,
        '--recipe-lock', RESTORED / 's2ut-recipe-lock.json', '--metadata', RESTORED / 'research-metadata.json',
        '--output', BASELINE, '--resume')
    baseline = json.loads(BASELINE.read_text(encoding='utf-8'))
    print({key:baseline[key] for key in ('checkpoint_updates','training_budget','selection_policy')})
    BACKBONE_CONFIG.write_text(json.dumps({'backbone':baseline['backbone']}),encoding='utf-8')
else:
    log_event('baseline-registration', 'skipped', LOGS, 'stage disabled or CPU audio preparation')

In [ ]:
if not PREPARE_AUDIO_PACKS and NEED_TEACHER:
    run('full-manifest', 'alignment_transfer_s2ut.benchmark_data', 'manifest',
        '--common', COMMON, '--prepared', PREPARED, '--output', MANIFEST, '--resume')
    run('import-units', 'alignment_transfer_s2ut.benchmark_data', 'units', '--manifest', MANIFEST,
        '--benchmark-units', BENCHMARK_UNITS, '--prepared', PREPARED,
        '--output', Path(ROOT) / 'units', '--resume')
else:
    log_event('speech-artifacts', 'skipped', LOGS, 'stage disabled or CPU audio preparation')

## 5. Teacher

T2TTの元train/dev hashと音声に対応するID・split・本文を照合してから、全層Attention、CTC alignment、時間座標への投影、品質診断を実行します。

In [ ]:
T2TT_ACTIVE_ROOT = Path(ROOT) / 'teacher-text-model'
T2TT_CHECKPOINT = T2TT_ACTIVE_ROOT / 'checkpoint_best.pt'
T2TT_ATTENTION_STAGE = 't2tt-attention-all-layers'
T2TT_ATTENTION_ROOT = Path(ROOT) / T2TT_ATTENTION_STAGE
CTC_STAGE = 'speech-text-alignment'
CTC_ROOT = Path(ROOT) / CTC_STAGE
TEACHER_STAGE = 'teacher-all-layers'
TEACHER_ROOT = Path(ROOT) / TEACHER_STAGE
TEACHER_AUDIT_STAGE = 'teacher-diagnostics'
TEACHER_AUDIT_ROOT = Path(ROOT) / TEACHER_AUDIT_STAGE
LAYER_FLAGS = ['--all-layers']
RESEARCH_CONDITIONS = [(method,label,globals()['RUN_' + method.upper()]) for method,label in METHOD_LABELS]
NEED_T2TT = NEED_TEACHER

In [ ]:
if not PREPARE_AUDIO_PACKS and NEED_TEACHER:
    teacher_stages = (T2TT_ATTENTION_STAGE, CTC_STAGE, TEACHER_STAGE)
    saved_teacher = all(any((Path(PERSISTENT) / 'stages' / stage).glob('*/snapshot.json')) for stage in teacher_stages)
    if not saved_teacher:
        if not RUN_TEACHER_ARTIFACTS:
            raise ValueError('初回はT2TTの評価を確認してRUN_TEACHER_ARTIFACTS=Trueにしてください')
        if not TEXT_DEV_METRICS.is_file() or json.loads(TEXT_DEV_METRICS.read_text(encoding='utf-8')).get('checkpoint_hash') != status['checkpoint_hash']:
            raise ValueError('EVALUATE_T2TT=Trueで選択中のT2TTのdev評価を完了してください')
    text_manifest = Path(ROOT) / 'teacher-text/manifest.jsonl'
    execute_json(PYTHON, '-m', 'alignment_transfer_s2ut.text_manifest', '--input', str(TEXT_MANIFEST),
        '--output', str(text_manifest), '--resume', name='teacher-text')
    snapshots = sorted(TEXT_BACKUP.glob('*/snapshot.json'), reverse=True)
    if not snapshots:
        raise FileNotFoundError('T2TTの検証済みsnapshotがありません')
    execute_json(PYTHON, '-m', 'alignment_transfer_s2ut.corpus_runtime', 'restore-teacher',
        '--snapshot', str(snapshots[0].parent), '--output', str(T2TT_ACTIVE_ROOT), '--updates', str(T2TT_MAX_UPDATES),
        '--source-manifest', str(text_manifest), '--speech-manifest', str(MANIFEST),
        '--backbone', str(BACKBONE_CONFIG), name='verify-teacher')
    run(T2TT_ATTENTION_STAGE, 'alignment_transfer_s2ut.extract_t2tt_attention', '--manifest', MANIFEST,
        '--checkpoint', T2TT_CHECKPOINT, '--output', T2TT_ATTENTION_ROOT, '--device', 'cuda',
        '--resume', *LAYER_FLAGS, '--teacher-training-manifest', text_manifest, gpu=True)
    run(CTC_STAGE, 'alignment_transfer_s2ut.align_speech_text', '--manifest', MANIFEST,
        '--config', 'configs/alignment/ctc.yaml', '--output', CTC_ROOT, '--device', 'cuda', '--resume', gpu=True)
else:
    log_event('teacher-extraction', 'skipped', LOGS, 'stage disabled or CPU audio preparation')

In [ ]:
if not PREPARE_AUDIO_PACKS and NEED_TEACHER:
    if NEED_T2TT:
        run(TEACHER_STAGE, 'alignment_transfer_s2ut.build_teacher_alignment', '--manifest', MANIFEST,
            '--mt-attention', T2TT_ATTENTION_ROOT, '--alignment', CTC_ROOT, '--units', Path(ROOT) / 'units',
            '--config', BACKBONE_CONFIG, '--output', TEACHER_ROOT, '--resume')
        run(TEACHER_AUDIT_STAGE, 'alignment_transfer_s2ut.diagnose_alignment', '--manifest', MANIFEST,
            '--teacher', TEACHER_ROOT, '--units', Path(ROOT) / 'units', '--config', BACKBONE_CONFIG,
            '--split', 'dev', '--max-samples', 500, '--heatmaps', 20, '--output', TEACHER_AUDIT_ROOT, '--overwrite', *LAYER_FLAGS)
    else:
        log_event('teacher-preparation', 'skipped', LOGS, 'teacher stage not requested')
else:
    log_event('teacher-projection', 'skipped', LOGS, 'stage disabled or CPU audio preparation')

## 6. top-p診断と校正

RUN_MASK_TOPP_DIAGNOSTICS=Trueで表を確認し、MASK_TOP_Pを0.8 / 0.9 / 0.95から選択します。RUN_ALIGNMENT_CALIBRATION=Trueでλ/τを校正します。

In [ ]:
if not PREPARE_AUDIO_PACKS and NEED_TEACHER:
    CALIBRATION_INPUTS = ['--baseline', str(BASELINE), '--manifest', str(MANIFEST),
        '--units', str(Path(ROOT) / 'units'), '--teacher', str(TEACHER_ROOT), '--batches', str(CALIBRATION_BATCHES)]
    topp_saved = any((Path(PERSISTENT) / 'stages' / ('research-v3-top-p' + SCOPE_SUFFIX)).glob('*/snapshot.json'))
    calibration_saved = any((Path(PERSISTENT) / 'stages' / ('research-v3-calibration' + SCOPE_SUFFIX)).glob('*/snapshot.json'))
    if RUN_MASK_TOPP_DIAGNOSTICS or topp_saved:
        run('research-v3-top-p' + SCOPE_SUFFIX, 'alignment_transfer_s2ut.alignment_calibration', 'top-p',
            *CALIBRATION_INPUTS, '--output', TOPP_REPORT, gpu=True)
    else:
        log_event('top-p-diagnostics', 'skipped', LOGS, 'RUN_MASK_TOPP_DIAGNOSTICS=False')
    if TOPP_REPORT.is_file():
        import pandas as pd
        import matplotlib.pyplot as plt
        from IPython.display import display, Image
        top_p_report = json.loads(TOPP_REPORT.read_text(encoding='utf-8'))
        display(pd.DataFrame([{'p':p, **{k:v for k,v in values.items() if k not in ('by_layer','by_source_length','histogram')}}
            for p,values in top_p_report['candidates'].items()]))
        for p,values in top_p_report['candidates'].items():
            print('p=', p, 'layer/source-length coverage')
            display(pd.DataFrame(values['by_layer']).T)
            display(pd.DataFrame(values['by_source_length']).T)
            plt.plot([(i+.5)/10 for i in range(10)], values['histogram'], label=p)
        plt.xlabel('Mask coverage'); plt.ylabel('Valid teacher rows'); plt.legend()
        plt.savefig(TOPP_ROOT / 'coverage.png'); plt.close()
        display(Image(filename=str(TOPP_ROOT / 'coverage.png')))
    if RUN_ALIGNMENT_CALIBRATION or calibration_saved:
        if MASK_TOP_P not in (.8,.9,.95):
            raise ValueError('top-p診断を確認し、MASK_TOP_Pを0.8 / 0.9 / 0.95へ明示設定してください')
        if not TOPP_REPORT.is_file():
            raise ValueError('先にtop-p diagnosticsを実行/復元してください')
        run('research-v3-calibration' + SCOPE_SUFFIX, 'alignment_transfer_s2ut.alignment_calibration', 'calibrate',
            *CALIBRATION_INPUTS, '--diagnostics', TOPP_REPORT, '--mask-top-p', MASK_TOP_P,
            '--output', CALIBRATION, gpu=True)
    else:
        log_event('alignment-calibration', 'skipped', LOGS, 'RUN_ALIGNMENT_CALIBRATION=False')
    if CALIBRATION.is_file():
        calibrated = json.loads(CALIBRATION.read_text(encoding='utf-8'))
        print({key:calibrated[key] for key in ('lambda_kl','lambda_mask','tau_kl','tau_mask','selected_top_p','optimizer_updates')})
else:
    log_event('calibration', 'skipped', LOGS, 'stage disabled or CPU audio preparation')

## 7. 転移3条件の学習

RUN_DISTILLATION、RUN_ANNEALED_DISTILLATION、RUN_MASK_DISTILLATIONで条件を選びます。各条件はBaselineと同じ400k予算・初期化を使います。全4条件は合計160万updatesです。

In [ ]:
if not PREPARE_AUDIO_PACKS:
    # 3条件は同一calibration artifact・全層・同じBaseline recipeを使う。
    for method, label, enabled in RESEARCH_CONDITIONS:
        if not enabled:
            log_event(method, 'skipped', LOGS, 'training switch is off')
            continue
        if not CALIBRATION.is_file() or MASK_TOP_P not in (.8,.9,.95):
            raise ValueError('top-p診断・明示設定・calibrationを完了してください')
        folder = RESEARCH_ROOT / method
        arguments = ['--baseline', str(BASELINE), '--output', str(folder), '--method', method,
            '--manifest', str(MANIFEST), '--units', str(Path(ROOT) / 'units'), '--teacher', str(TEACHER_ROOT),
            '--calibration', str(CALIBRATION), '--mask-top-p', str(MASK_TOP_P), '--all-layers', '--resume']
        plan = execute_json(PYTHON, '-m', 'alignment_transfer_s2ut.benchmark', 'plan', *arguments, name='plan-' + method)
        print({'condition':label, 'training_budget':plan['training_budget'], 'initialization':plan['initialization'],
            'selection':plan['selection_policy'], 'calibration_hash':plan['transfer']['calibration_hash']})
        run('research-v3-' + method + SCOPE_SUFFIX, 'alignment_transfer_s2ut.benchmark', 'train', *arguments, gpu=True)
else:
    log_event('transfer-training', 'skipped', LOGS, 'stage disabled or CPU audio preparation')

In [ ]:
if not PREPARE_AUDIO_PACKS and NEED_TEACHER:
    conditions = [('baseline', 'Baseline', BASELINE)]
    for method, label, enabled in RESEARCH_CONDITIONS:
        folder = RESEARCH_ROOT / method
        if enabled:
            record = RESEARCH_ROOT / 'registrations' / (method + '.json')
            run('research-v3-register-' + method, 'alignment_transfer_s2ut.benchmark', 'register-transfer',
                '--run', folder, '--checkpoint', folder / 'checkpoints/checkpoint_best.pt', '--output', record, '--resume')
            conditions.append((method,label,record))
else:
    log_event('transfer-registration', 'skipped', LOGS, 'stage disabled or CPU audio preparation')

## 8. 学習診断と評価への接続

RUN_ALIGNMENT_DIAGNOSTICS=Trueで層別診断・可視化を表示します。推論・評価は同じEXPERIMENT_IDで [evaluate.ipynb](evaluate.ipynb) を開きます。

In [ ]:
if not PREPARE_AUDIO_PACKS and RUN_ALIGNMENT_DIAGNOSTICS:
    # 生成とは独立したteacher-forced dev診断。teacherを生成器へ渡さない。
    if globals().get('conditions') and CALIBRATION.is_file():
        import pandas as pd
        from IPython.display import display, Image
        completed = []
        for name, label, record in conditions:
            output = RESEARCH_ROOT / 'alignment-diagnostics' / name / 'diagnostics.json'
            run('research-v3-alignment-' + name + SCOPE_SUFFIX, 'alignment_transfer_s2ut.alignment_report', 'diagnose',
                '--baseline', BASELINE, '--record', record, '--manifest', MANIFEST, '--units', Path(ROOT) / 'units',
                '--teacher', TEACHER_ROOT, '--calibration', CALIBRATION, '--samples', 32, '--output', output, gpu=True)
            result = json.loads(output.read_text(encoding='utf-8'))
            print(label, 'KL mean/worst:', result['kl_mean'], result['kl_worst'])
            display(pd.DataFrame({'KL':result['per_layer_kl'], 'Mask loss':result['per_layer_mask_loss'],
                'Region mass':result['teacher_region_mass'], 'Worst-layer weight':result['worst_layer_weight_distribution']}))
            if name != 'baseline':
                dev_pair = next(json.loads(line)['pair_id'] for line in MANIFEST.read_text(encoding='utf-8').splitlines()
                    if line.strip() and json.loads(line)['split']=='dev')
                visual = RESEARCH_ROOT / 'attention-visualization' / name
                run('research-v3-visualize-' + name + SCOPE_SUFFIX, 'alignment_transfer_s2ut.visualize_attention',
                    '--manifest', MANIFEST, '--checkpoint', RESEARCH_ROOT / name / 'checkpoints/checkpoint_best.pt',
                    '--units', Path(ROOT) / 'units', '--teacher', TEACHER_ROOT, '--pair-id', dev_pair,
                    '--device', 'cuda', '--output', visual, '--overwrite', gpu=True)
                for path in visual.glob('*.png'):
                    display(Image(filename=str(path)))
            if name != 'baseline':
                completed.append(RESEARCH_ROOT / name)
        if completed:
            curves_root = RESEARCH_ROOT / 'curves'
            run('research-v3-curves' + SCOPE_SUFFIX, 'alignment_transfer_s2ut.alignment_report', 'curves',
                '--runs', *completed, '--output', curves_root)
            for folder in completed:
                display(Image(filename=str(curves_root / (folder.name + '.png'))))
    else:
        log_event('layer-diagnostics', 'skipped', LOGS, 'complete calibration and common evaluation first')
else:
    log_event('layer-diagnostics', 'skipped', LOGS, 'stage disabled or CPU audio preparation')